In [1]:
import pandas as pd

In [15]:
df = pd.read_excel("fuel_df_15_09_26.xlsx")

In [16]:
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes.value_counts())
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDescribe (numeric columns):")
print(df.describe())

Shape: (2689, 90)

Dtypes:
str               82
float64            6
datetime64[us]     2
Name: count, dtype: int64

Missing values per column:
Gross_Weight_Conditional_At_Touchdown_kg    2679
Holding_Duration_sec                        2653
Speedbrake_With_Power_On_Duration_sec       2635
Speedbrake_With_Gear_Down_Duration_sec_     2628
APU_On_In_Flight_Duration_sec               2597
                                            ... 
Aircraft_Registration                          0
Departure_Airport_ICAO                         0
Departure_Airport_IATA                         0
Fleet                                          0
Takeoff_Date_time                              0
Length: 90, dtype: int64

Duplicate rows: 802

Describe (numeric columns):
          Flight_ID                        Date           Takeoff_Date_time  \
count  2.686000e+03                        2686                        2689   
mean   7.239391e+07  2026-05-26 19:27:39.270290  2026-05-27 07:19:32.835627   
min  

In [17]:
df["Total_Fuel_Burn_kg"] = (
    df["Total_Fuel_Burn_kg"]
    .astype(str)
    .str.extract(r"([-+]?\d*\.?\d+)")[0]
    .astype(float)
)

print(df["Total_Fuel_Burn_kg"].head())
print(df["Total_Fuel_Burn_kg"].dtype)

0    46901.0
1     1960.0
2     3138.0
3     7588.0
4      863.0
Name: Total_Fuel_Burn_kg, dtype: float64
float64


In [18]:
for col in df.select_dtypes(include="object").columns:
    if col not in ["Flight_Number", "Fleet", "Aircraft_Type", "Aircraft_Registration",
                   "Departure_Airport_ICAO", "Departure_Airport_IATA",
                   "Arrival_Airport_ICAO", "Arrival_Airport_IATA", "Date_time"]:
        df[col] = df[col].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0].astype(float)

print(df.dtypes.value_counts())

C:\Users\hp\AppData\Local\Temp\ipykernel_17892\136018685.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


float64           80
str                8
datetime64[us]     2
Name: count, dtype: int64


In [19]:
print(df.select_dtypes(include="object").columns.tolist())

df["Route"] = df["Departure_Airport_IATA"] + "-" + df["Arrival_Airport_IATA"]
print(df["Route"].head())

['Flight_Number', 'Fleet', 'Aircraft_Type', 'Aircraft_Registration', 'Departure_Airport_ICAO', 'Departure_Airport_IATA', 'Arrival_Airport_ICAO', 'Arrival_Airport_IATA']
0    LHR-KGL
1    KGL-EBB
2    NBO-KGL
3    KGL-JIB
4    KGL-EBB
Name: Route, dtype: str


C:\Users\hp\AppData\Local\Temp\ipykernel_17892\2313174142.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include="object").columns.tolist())


In [20]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Flight_ID:", df.duplicated(subset=["Flight_ID"]).sum())
print("Duplicate Flight_Number + Date:", df.duplicated(subset=["Flight_Number", "Date"]).sum())

Exact duplicate rows: 802
Duplicate Flight_ID: 805
Duplicate Flight_Number + Date: 1176


In [21]:
before = len(df)
df = df.drop_duplicates(keep="first")
print(f"Rows before: {before}")
print(f"Rows after: {len(df)}")
print(f"Dropped: {before - len(df)}")

Rows before: 2689
Rows after: 1887
Dropped: 802


In [22]:
print(df["Fleet"].value_counts())

Fleet
RWD-B737-NG      1078
RWD-DHC-8-400     479
RWD-A330          283
RWD-B737F-NG       47
Name: count, dtype: int64


In [24]:
df.info()

<class 'pandas.DataFrame'>
Index: 1887 entries, 0 to 2688
Data columns (total 91 columns):
 #   Column                                         Non-Null Count  Dtype         
---  ------                                         --------------  -----         
 0   Flight_ID                                      1884 non-null   float64       
 1   Flight_Number                                  1872 non-null   str           
 2   Date                                           1884 non-null   datetime64[us]
 3   Takeoff_Date_time                              1887 non-null   datetime64[us]
 4   Fleet                                          1887 non-null   str           
 5   Aircraft_Type                                  1887 non-null   str           
 6   Aircraft_Registration                          1887 non-null   str           
 7   Departure_Airport_ICAO                         1887 non-null   str           
 8   Departure_Airport_IATA                         1887 non-null   str        

In [48]:
pd.set_option("display.max_rows", None)
corr = df.corr(numeric_only=True)["Total_Fuel_Burn_kg"].drop("Total_Fuel_Burn_kg")
print(corr.sort_values(ascending=False))

Fuel_Qty_At_Liftoff_kg                           0.985389
Fuel_Qty_Off_Blocks_kg                           0.976008
Distance_Flown_In_Cruise_NM                      0.951901
Great_Circle_Distance_NM                         0.949861
Total_Distance_Flown_NM                          0.949186
Total_Distance_Travelled_NM                      0.949073
Engine_2-Running_Duration_sec                    0.937707
Engine_1_Running_Duration_sec                    0.934304
Engine_Running_Duration_Total_sec                0.931472
Gross_Weight_At_Liftoff_kg                       0.890441
Gross_Weight_At_Touchdown_kg                     0.812108
Gross_Weight_Conditional_At_Touchdown_kg         0.788040
Max_Eng_EPR_During_Cruise                        0.775701
Distance_Flown_In_Climb_NM                       0.683091
Max_Eng_EPR_During_Takeoff                       0.680615
Taxi_Out_Fuel_Burn_kg                            0.655238
Distance_Flown_In_Descent_NM                     0.653110
Max_Airspeed_A

In [50]:
cols = ["Total_Distance_Flown_NM", "Great_Circle_Distance_NM", "Gross_Weight_At_Liftoff_kg",
        "Max_Tailwind_During_Takeoff_kt", "Max_Static_Air_Temperature_degree_celsius"]

print(df[cols].corr())

                                           Total_Distance_Flown_NM  \
Total_Distance_Flown_NM                                   1.000000   
Great_Circle_Distance_NM                                  0.994016   
Gross_Weight_At_Liftoff_kg                                0.749567   
Max_Tailwind_During_Takeoff_kt                            0.305546   
Max_Static_Air_Temperature_degree_celsius                -0.024016   

                                           Great_Circle_Distance_NM  \
Total_Distance_Flown_NM                                    0.994016   
Great_Circle_Distance_NM                                   1.000000   
Gross_Weight_At_Liftoff_kg                                 0.748158   
Max_Tailwind_During_Takeoff_kt                             0.298842   
Max_Static_Air_Temperature_degree_celsius                 -0.034259   

                                           Gross_Weight_At_Liftoff_kg  \
Total_Distance_Flown_NM                                      0.749567   
Great_

In [51]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np

cols = ["Total_Distance_Flown_NM", "Great_Circle_Distance_NM",
        "Gross_Weight_At_Liftoff_kg", "Max_Tailwind_During_Takeoff_kt"]

X = df[cols].dropna().copy()
X["log_distance_flown"] = np.log(X["Total_Distance_Flown_NM"])
X["log_gcd"] = np.log(X["Great_Circle_Distance_NM"])
X["log_weight"] = np.log(X["Gross_Weight_At_Liftoff_kg"])
X = X[["log_distance_flown", "log_gcd", "log_weight", "Max_Tailwind_During_Takeoff_kt"]]
X = sm.add_constant(X)

for i, col in enumerate(X.columns):
    if col == "const":
        continue
    vif = variance_inflation_factor(X.values, i)
    print(f"{col:35s} VIF = {vif:.2f}")

log_distance_flown                  VIF = 95.81
log_gcd                             VIF = 93.10
log_weight                          VIF = 1.55
Max_Tailwind_During_Takeoff_kt      VIF = 1.02


In [61]:
model_cols = [
    "Flight_ID", "Flight_Number", "Date", "Fleet", "Aircraft_Registration", "Route",
    "Total_Distance_Flown_NM",
    "Great_Circle_Distance_NM",
    "Gross_Weight_At_Liftoff_kg",
    "Max_Tailwind_During_Takeoff_kt",
    "Max_Static_Air_Temperature_degree_celsius",
    "Total_Fuel_Burn_kg",
]

model_df = df[model_cols].copy()

print(model_df.shape)
print(model_df.isna().sum())

(1887, 12)
Flight_ID                                      3
Flight_Number                                 15
Date                                           3
Fleet                                          0
Aircraft_Registration                          0
Route                                          1
Total_Distance_Flown_NM                        4
Great_Circle_Distance_NM                       5
Gross_Weight_At_Liftoff_kg                   482
Max_Tailwind_During_Takeoff_kt                 4
Max_Static_Air_Temperature_degree_celsius      3
Total_Fuel_Burn_kg                             3
dtype: int64


In [62]:
# Fallback chain: aircraft+route median -> fleet median (no global fallback -
# a global median would fabricate values for a fleet with no real data at all)
model_df["Gross_Weight_At_Liftoff_kg"] = (
    model_df.groupby(["Aircraft_Registration", "Route"])["Gross_Weight_At_Liftoff_kg"]
    .transform(lambda x: x.fillna(x.median()))
)

still_missing = model_df["Gross_Weight_At_Liftoff_kg"].isna().sum()
print(f"Still missing after aircraft+route median: {still_missing}")

if still_missing > 0:
    model_df["Gross_Weight_At_Liftoff_kg"] = (
        model_df.groupby("Fleet")["Gross_Weight_At_Liftoff_kg"]
        .transform(lambda x: x.fillna(x.median()))
    )
    still_missing = model_df["Gross_Weight_At_Liftoff_kg"].isna().sum()
    print(f"Still missing after fleet median: {still_missing}")

if still_missing > 0:
    affected_fleets = model_df.loc[
        model_df["Gross_Weight_At_Liftoff_kg"].isna(), "Fleet"
    ].value_counts()
    print(f"\n{still_missing} rows have no usable weight data even at the fleet level.")
    print("These fleets have essentially no real weight sensor data:")
    print(affected_fleets)
    print("\nThese rows will be dropped rather than filled with a fabricated value.")
    model_df = model_df.dropna(subset=["Gross_Weight_At_Liftoff_kg"])

print("\nFinal shape:", model_df.shape)
print("Final missing count:", model_df["Gross_Weight_At_Liftoff_kg"].isna().sum())
print("\nFleet counts:")
print(model_df["Fleet"].value_counts())

Still missing after aircraft+route median: 480
Still missing after fleet median: 479

479 rows have no usable weight data even at the fleet level.
These fleets have essentially no real weight sensor data:
Fleet
RWD-DHC-8-400    479
Name: count, dtype: int64

These rows will be dropped rather than filled with a fabricated value.

Final shape: (1408, 12)
Final missing count: 0

Fleet counts:
Fleet
RWD-B737-NG     1078
RWD-A330         283
RWD-B737F-NG      47
Name: count, dtype: int64


In [63]:
print(model_df.loc[model_df["Gross_Weight_At_Liftoff_kg"] == 62556, "Fleet"].value_counts())

Series([], Name: count, dtype: int64)


In [64]:
before = len(model_df)
model_df = model_df[model_df["Fleet"] != "RWD-DHC-8-400"].copy()

print(f"Rows before: {before}")
print(f"Rows after: {len(model_df)}")
print(f"Removed: {before - len(model_df)}")

print("\nRemaining missing values:")
print(model_df.isna().sum())

print("\nFleet counts:")
print(model_df["Fleet"].value_counts())

Rows before: 1408
Rows after: 1408
Removed: 0

Remaining missing values:
Flight_ID                                     2
Flight_Number                                15
Date                                          2
Fleet                                         0
Aircraft_Registration                         0
Route                                         1
Total_Distance_Flown_NM                       3
Great_Circle_Distance_NM                      4
Gross_Weight_At_Liftoff_kg                    0
Max_Tailwind_During_Takeoff_kt                3
Max_Static_Air_Temperature_degree_celsius     2
Total_Fuel_Burn_kg                            2
dtype: int64

Fleet counts:
Fleet
RWD-B737-NG     1078
RWD-A330         283
RWD-B737F-NG      47
Name: count, dtype: int64


In [65]:
missing_mask = model_df.isna().any(axis=1)
print(f"Rows with at least one missing value: {missing_mask.sum()}")

print("\nFleet breakdown of these rows:")
print(model_df[missing_mask]["Fleet"].value_counts())

print("\nThe rows themselves:")
print(model_df[missing_mask])

Rows with at least one missing value: 18

Fleet breakdown of these rows:
Fleet
RWD-B737-NG     17
RWD-B737F-NG     1
Name: count, dtype: int64

The rows themselves:
       Flight_ID Flight_Number       Date         Fleet Aircraft_Registration  \
42           NaN           108        NaT   RWD-B737-NG                9XR-WY   
114   71280572.0           NaN 2026-04-16  RWD-B737F-NG                9XR-WW   
303   71496146.0           NaN 2026-04-01   RWD-B737-NG                9XR-WY   
413   71612397.0           423 2026-04-30   RWD-B737-NG                9XR-WT   
458          NaN           102        NaT   RWD-B737-NG                9XR-WU   
556   72136103.0           NaN 2026-05-18   RWD-B737-NG                9XR-WF   
669   72271227.0           NaN 2026-05-24   RWD-B737-NG                9XR-WG   
673   72271232.0           NaN 2026-05-24   RWD-B737-NG                9XR-WG   
680   72271248.0           NaN 2026-05-23   RWD-B737-NG                9XR-WF   
681   72271251.0         

In [66]:
# Drop the two rows with no Flight_ID at all (genuinely broken rows)
model_df = model_df.drop(index=[42, 458])

# Build a Route -> most common Flight_Number lookup from rows that have both
route_to_flight = (
    model_df.dropna(subset=["Route", "Flight_Number"])
    .groupby("Route")["Flight_Number"]
    .agg(lambda x: x.mode().iloc[0])  # most frequent flight number for that route
)

# Fill missing Flight_Number using the route lookup
missing_mask = model_df["Flight_Number"].isna()
model_df.loc[missing_mask, "Flight_Number"] = model_df.loc[missing_mask, "Route"].map(route_to_flight)

print("Remaining missing values:")
print(model_df.isna().sum())
print("\nRows still missing Flight_Number (no route to look up, or route unseen elsewhere):")
print(model_df[model_df["Flight_Number"].isna()])

Remaining missing values:
Flight_ID                                    0
Flight_Number                                1
Date                                         0
Fleet                                        0
Aircraft_Registration                        0
Route                                        1
Total_Distance_Flown_NM                      1
Great_Circle_Distance_NM                     2
Gross_Weight_At_Liftoff_kg                   0
Max_Tailwind_During_Takeoff_kt               1
Max_Static_Air_Temperature_degree_celsius    0
Total_Fuel_Burn_kg                           0
dtype: int64

Rows still missing Flight_Number (no route to look up, or route unseen elsewhere):
       Flight_ID Flight_Number       Date        Fleet Aircraft_Registration  \
2625  75199917.0           NaN 2026-08-14  RWD-B737-NG                9XR-WY   

     Route  Total_Distance_Flown_NM  Great_Circle_Distance_NM  \
2625   NaN                      NaN                       NaN   

      Gross_Weight_At

In [67]:
model_df = model_df.dropna()

print("Final missing values:")
print(model_df.isna().sum())
print("\nFinal shape:", model_df.shape)
print("\nFleet counts:")
print(model_df["Fleet"].value_counts())

Final missing values:
Flight_ID                                    0
Flight_Number                                0
Date                                         0
Fleet                                        0
Aircraft_Registration                        0
Route                                        0
Total_Distance_Flown_NM                      0
Great_Circle_Distance_NM                     0
Gross_Weight_At_Liftoff_kg                   0
Max_Tailwind_During_Takeoff_kt               0
Max_Static_Air_Temperature_degree_celsius    0
Total_Fuel_Burn_kg                           0
dtype: int64

Final shape: (1404, 12)

Fleet counts:
Fleet
RWD-B737-NG     1074
RWD-A330         283
RWD-B737F-NG      47
Name: count, dtype: int64


In [70]:
model_df.to_excel("model_df_clean.xlsx", index=False)
print("Saved. Shape:", model_df.shape)

Saved. Shape: (1404, 12)


In [69]:
model_df.describe()

,Flight_ID,Date,Total_Distance_Flown_NM,Great_Circle_Distance_NM,Gross_Weight_At_Liftoff_kg,Max_Tailwind_During_Takeoff_kt,Max_Static_Air_Temperature_degree_celsius,Total_Fuel_Burn_kg
count,1.404000e+03,1404,1404.000000,1404.000000,1404.000000,1404.000000,1404.000000,1404.000000
mean,7.264815e+07,2026-06-04 07:24:06.153846,1218.176425,1071.724145,87241.477208,-0.998575,38.851709,10822.646724
min,6.971678e+07,2026-03-07 00:00:00,44.000000,39.200000,47392.000000,-16.000000,12.200000,720.000000
25%,7.164052e+07,2026-05-01 00:00:00,225.975000,213.800000,57044.500000,-4.000000,25.775000,2205.000000
50%,7.207389e+07,2026-05-17 00:00:00,628.500000,601.100000,62568.500000,-1.000000,31.000000,4362.500000
75%,7.474582e+07,2026-08-12 00:00:00,1795.025000,1455.000000,74298.000000,2.000000,58.500000,11799.250000
max,7.525124e+07,2026-08-28 00:00:00,4261.200000,3567.000000,228376.000000,12.000000,58.500000,55116.000000
std,1.451229e+06,NaN,1182.218719,1043.330903,52885.673172,4.345224,15.518391,13776.012091
